# 45 — Beam singles λ = 0 / 0.4 on the bf16 validated pool (completes nb31's 2×2)

Reconstitutes notebook 31's greedy×beam × λ single-concept design on the
**validated bf16 pool** instead of the legacy AWQ 27-concept set. nb44
supplied the greedy half; this notebook adds the beam rows, so the full
2×2 is finally on validated guides. Two questions this closes:

1. **Beam-multiplier reconfirmation** (a report Future-Work item): does
   beam's ~4.5× single-concept advantage over greedy — measured on the
   pre-validation set — hold on validated guides?
2. **Does λ stay "free" under beam** on strong guides, given that nb44
   showed λ became a *trade-off* under greedy once guides were validated?

40 pool concepts × beam × λ{0, 0.4} × 10 prompts. Greedy rows reused from
nb44 for the full table. True beam (k=4, pool k²=16), batch_size=4.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS, K, STEPS = 3, 4, 24
LAMBDAS = [0.0, 0.4]
BEAM_BATCH = 4
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_KEY = "llama-3.1-8b-bf16"

POOL = json.loads(Path("resources/32_concept_pool_bf16.json").read_text())
CONCEPTS = [r["synset"] for r in POOL["pool"]]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

LOG_PATH = Path("resources/45_singles_beam_bf16.jsonl")
LOG_PATH.unlink(missing_ok=True)
print(f"{len(CONCEPTS)} pool concepts x beam x {len(LAMBDAS)} lambda")

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    device_map="auto", id=MODEL_ID, torch_dtype=torch.bfloat16
)
prompts = [chat(u, a) for u, a in QUESTIONS]
harness = EvaluationHarness(
    fur, LOG_PATH, min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
)
baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)
concept = {n: fur.get_concept_cached(n, MIN_LEMMAS, MAX_TOKENS) for n in CONCEPTS}
print(f"{len(concept)} concepts built")

for name in CONCEPTS:
    c = concept[name]
    for lam in LAMBDAS:
        torch.cuda.empty_cache()
        texts, probe = fur.generate_with_topk_beam_guide(
            prompts,
            concepts=[c],
            k=K,
            steps=STEPS,
            fluency_weight=lam,
            batch_size=BEAM_BATCH,
        )
        harness.evaluate(
            prompts,
            texts,
            [name],
            config={
                "model": MODEL_KEY,
                "method": "single",
                "search": "beam",
                "lam": lam,
                "concept": name,
            },
            probe=probe,
            baseline_texts=baseline_texts,
            concepts={name: c},
            angles=True,
        )
    print(f"{name}: done")

In [ ]:
def load(path, search):
    rows = []
    for line in Path(path).read_text().strip().split("\n"):
        r = json.loads(line)
        c = r["config"]
        name = c["concept"]
        rows.append(
            {
                "search": search,
                "lam": c["lam"],
                "delta": r["expression"][name]["delta"],
                "d_cos": r["angles"][name]["delta"]["token_cos"],
                "ppl_ratio": r["ppl_ratio"],
            }
        )
    return pd.DataFrame(rows)


beam = load(LOG_PATH, "beam")
greedy = load("resources/44_singles_lambda_bf16.jsonl", "greedy")
df = pd.concat([greedy, beam], ignore_index=True)

summary = (
    df.groupby(["search", "lam"])
    .agg(
        delta=("delta", "mean"),
        d_cos=("d_cos", "mean"),
        positive=("delta", lambda s: (s > 0).mean()),
        ppl_ratio=("ppl_ratio", "mean"),
        n=("delta", "size"),
    )
    .reset_index()
)
summary.to_csv("resources/45_singles_2x2_bf16_summary.csv", index=False)
print("FULL 2x2 on VALIDATED bf16 pool:")
print(summary.round(4).to_string(index=False))

g0 = summary[(summary.search == "greedy") & (summary.lam == 0.0)]["delta"].iloc[0]
b0 = summary[(summary.search == "beam") & (summary.lam == 0.0)]["delta"].iloc[0]
b4 = summary[(summary.search == "beam") & (summary.lam == 0.4)]["delta"].iloc[0]
print()
print(f"  beam-multiplier (beam λ0 / greedy λ0): {b0/g0:.2f}x  [nb31 legacy AWQ: 4.5x]")
print(
    f"  beam λ dial: Δ {b0:.3f} -> {b4:.3f} ({b4/b0:.2f}x)  ppl "
    f"{summary[(summary.search=='beam')&(summary.lam==0.0)]['ppl_ratio'].iloc[0]:.1f} -> "
    f"{summary[(summary.search=='beam')&(summary.lam==0.4)]['ppl_ratio'].iloc[0]:.1f}"
)
print("  -> is λ still ~free under beam on validated guides?")

In [ ]:
cells = [("greedy", 0.0), ("greedy", 0.4), ("beam", 0.0), ("beam", 0.4)]
labels = [f"{srch}\nλ={lm}" for srch, lm in cells]
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, (col, lab) in zip(
    ax,
    (
        ("d_cos", "token-cosine Δ"),
        ("delta", "expression Δ"),
        ("ppl_ratio", "ppl ratio"),
    ),
):
    vals = [
        summary[(summary.search == srch) & (summary.lam == lm)][col].iloc[0]
        for srch, lm in cells
    ]
    a.bar(labels, vals, color=["tab:gray", "tab:cyan", "tab:blue", "tab:purple"])
    a.set_ylabel(lab)
    a.axhline(0, color="gray", lw=0.6)
ax[2].axhline(2.5, color="red", ls=":", lw=0.8)
plt.suptitle("Singles 2×2 (search × λ) on the bf16 validated pool")
plt.tight_layout()
plt.savefig("resources/45_singles_2x2_bf16.png", dpi=150)
plt.show()